# Severity conditional on a recorded crash

Real published results from DGT, INE and Eurostat. Author: Javier Saguar.

See [methodology](../docs/methodology.md), [sources](../docs/sources.md) and [limitations](../docs/limitations.md).

Reusable implementations live in `src/`; reproduce artifacts using the CLI before rerunning these explorations.

In [1]:
from pathlib import Path
import json
import pandas as pd
from ebdi.metrics.index import composite
from ebdi.utils.io import read_yaml
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
TABLES = ROOT / "outputs/tables"
spain = pd.read_csv(TABLES / "spain_metrics.csv", dtype={"province_code": str})


## Valid target, selection protocol and negative class

In [2]:
card = json.loads((TABLES / "model_card.json").read_text())
assert card["personal_accident_risk"] is False
print(card["target"])
print(card["selection_criterion"])
print("Predictors:", card["features"])
print(pd.read_csv(TABLES / "model_validation.csv").to_string(index=False))

Recorded injury crash has >=1 fatality or hospitalized injury at 30 days
2023 validation log loss; 2024 never used for model selection
Predictors: ['COD_PROVINCIA', 'ZONA_AGRUPADA', 'TIPO_VIA', 'TIPO_ACCIDENTE', 'NUDO', 'CONDICION_METEO', 'CONDICION_ILUMINACION', 'CONDICION_FIRME', 'DIA_SEMANA', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
                 model           split      n  positive_n  prevalence  roc_auc   pr_auc  log_loss    brier  threshold  precision   recall    tn  fp   fn  tp
                 dummy validation_2023 101306        9801    0.096746 0.500000 0.096746  0.317995 0.087407        0.5   0.000000 0.000000 91505   0 9801   0
              logistic validation_2023 101306        9801    0.096746 0.693791 0.206842  0.296191 0.083029        0.5   0.534884 0.002347 91485  20 9778  23
hist_gradient_boosting validation_2023 101306        9801    0.096746 0.700715 0.222349  0.293679 0.082363        0.5   0.591716 0.010203 91436  69 9701 100


## Untouched 2024 evaluation: compare all three candidates

In [3]:
metrics = pd.read_csv(TABLES / "model_metrics.csv")
assert metrics.n.eq(101996).all()
assert metrics.positive_n.eq(10019).all()
print(metrics.to_string(index=False))
print("Low recall at the fixed 0.5 threshold prevents interpreting AUC as screening performance.")

                 model     split  selected_on_validation      n  positive_n  prevalence  roc_auc   pr_auc  log_loss    brier  threshold  precision   recall    tn  fp    fn  tp
                 dummy test_2024                   False 101996       10019    0.098229 0.500000 0.098229  0.321254 0.088594        0.5   0.000000 0.000000 91977   0 10019   0
              logistic test_2024                   False 101996       10019    0.098229 0.689167 0.212135  0.299538 0.084078        0.5   0.394737 0.001497 91954  23 10004  15
hist_gradient_boosting test_2024                    True 101996       10019    0.098229 0.703037 0.233523  0.295697 0.083099        0.5   0.682353 0.011578 91923  54  9903 116
Low recall at the fixed 0.5 threshold prevents interpreting AUC as screening performance.


## Calibration has bin sample sizes; attribution is not causal

In [4]:
print(pd.read_csv(TABLES / "model_calibration.csv").to_string(index=False))
print(pd.read_csv(TABLES / "model_permutation_importance.csv").head(8).to_string(index=False))
if (TABLES / "model_shap.json").exists():
    shap = json.loads((TABLES / "model_shap.json").read_text())
    assert shap["maximum_additivity_error"] < 1e-5
    assert not shap["causal_interpretation"]
    print(shap)
    print(pd.read_csv(TABLES / "model_shap_importance.csv").head(8).to_string(index=False))

 mean_prediction  observed_frequency      n                  model
        0.094523            0.098229 101996                  dummy
        0.057996            0.060586  65428               logistic
        0.140571            0.134926  28082               logistic
        0.237953            0.227591   6560               logistic
        0.337654            0.382225   1609               logistic
        0.437384            0.512545    279               logistic
        0.530237            0.388889     36               logistic
        0.604104            0.500000      2               logistic
        0.059617            0.058485  65401 hist_gradient_boosting
        0.137851            0.140269  29921 hist_gradient_boosting
        0.236602            0.244549   4678 hist_gradient_boosting
        0.344197            0.367560   1344 hist_gradient_boosting
        0.439487            0.504149    482 hist_gradient_boosting
        0.533170            0.628788    132 hist_gradient_boos